In [2]:
import numpy as np

# Structure:
# [[TN, FP],
#  [FN, TP]]
confusion_matrices_cctx = np.array([
    [[174, 0],
    [3, 10]],
    [[173, 1],
     [2, 11]],
    [[172, 2],
     [1, 12]],
    [[169, 5],
     [1, 12]],
    [[173, 1],
     [4, 9]]
])

confusion_matrices_single = np.array([
    [[129, 9],
     [0, 35]],
    [[131, 7],
     [1, 34]],
    [[128, 10],
     [0, 35]],
    [[130, 8],
     [0, 35]],
    [[131, 7],
     [0, 35]]
])

In [7]:
def metrics_from_cm(cm):
    """
    Returns dict of arrays (length N) for accuracy, precision/recall/f1 per class.
    """
    tn = cm[:, 0, 0]
    fp = cm[:, 0, 1]
    fn = cm[:, 1, 0]
    tp = cm[:, 1, 1]

    total = tn + fp + fn + tp
    accuracy = (tp + tn) / total

    # Positive class
    precision_pos = tp / (tp + fp)
    recall_pos    = tp / (tp + fn)
    f1_pos        = 2 * precision_pos * recall_pos / (precision_pos + recall_pos)

    # Negative class (treat "negative" as the positive class of a flipped problem)
    precision_neg = tn / (tn + fn)
    recall_neg    = tn / (tn + fp)
    f1_neg        = 2 * precision_neg * recall_neg / (precision_neg + recall_neg)

    return {
        "accuracy": accuracy,
        "precision_neg": precision_neg, "recall_neg": recall_neg, "f1_neg": f1_neg,
        "precision_pos": precision_pos, "recall_pos": recall_pos, "f1_pos": f1_pos,
    }

def report(name, metrics_dict):
    print(f"--- {name} ---")
    for k, v in metrics_dict.items():
        print(f"{k:15s} mean={np.mean(v):.4f}  std={np.std(v):.4f}")
    print()

# Per-scope
m_cctx = metrics_from_cm(confusion_matrices_cctx)
m_single = metrics_from_cm(confusion_matrices_single)
report("CCTX", m_cctx)
report("SINGLE", m_single)

--- CCTX ---
accuracy        mean=0.9786  std=0.0068
precision_neg   mean=0.9875  std=0.0065
recall_neg      mean=0.9897  std=0.0099
f1_neg          mean=0.9885  std=0.0037
precision_pos   mean=0.8759  std=0.0969
recall_pos      mean=0.8308  std=0.0897
f1_pos          mean=0.8442  std=0.0440

--- SINGLE ---
accuracy        mean=0.9514  std=0.0059
precision_neg   mean=0.9985  std=0.0030
recall_neg      mean=0.9406  std=0.0085
f1_neg          mean=0.9686  std=0.0040
precision_pos   mean=0.8100  std=0.0209
recall_pos      mean=0.9943  std=0.0114
f1_pos          mean=0.8925  std=0.0114



In [8]:

# ---- Option A: sample-weighted global (recommended) ----
# Sum matching experiment-index CMs elementwise -> 5 "combined" CMs
combined_cm = confusion_matrices_cctx + confusion_matrices_single
m_combined = metrics_from_cm(combined_cm)
report("GLOBAL (sample-weighted, sum of CMs per experiment)", m_combined)

--- GLOBAL (sample-weighted, sum of CMs per experiment) ---
accuracy        mean=0.9656  std=0.0028
precision_neg   mean=0.9922  std=0.0039
recall_neg      mean=0.9679  std=0.0067
f1_neg          mean=0.9799  std=0.0018
precision_pos   mean=0.8217  std=0.0268
recall_pos      mean=0.9500  std=0.0250
f1_pos          mean=0.8805  std=0.0067



In [9]:

# ---- Option B: simple average of the two scopes' metrics (unweighted) ----
m_unweighted = {
    k: (m_cctx[k] + m_single[k]) / 2
    for k in m_cctx
}
report("GLOBAL (unweighted average of scope metrics)", m_unweighted)

--- GLOBAL (unweighted average of scope metrics) ---
accuracy        mean=0.9650  std=0.0028
precision_neg   mean=0.9930  std=0.0035
recall_neg      mean=0.9651  std=0.0066
f1_neg          mean=0.9786  std=0.0017
precision_pos   mean=0.8429  std=0.0490
recall_pos      mean=0.9125  std=0.0447
f1_pos          mean=0.8683  std=0.0175

